# 11. Recommender Systems

**Machine Learning Fundamentals and Predictive Analytics — Notebook 11 of 11**

The final notebook in the module ties together ideas from across the course — similarity from
KNN (Notebook 4), latent factors from PCA/NMF (statistics module and Notebook 9), and honest
evaluation from every notebook before this one — to answer a deceptively simple question:
*what will this person like next?*

### What you will learn

1. The recommendation problem, and the **utility matrix**
2. **Content-based filtering**: recommend by item similarity
3. **Collaborative filtering**: recommend by user/item similarity in behaviour
4. **Matrix factorisation** (SVD / ALS): latent factors, learned from data
5. The **cold-start** problem, and how to mitigate it
6. **Evaluation**: RMSE-based vs ranking-based metrics
7. **Implicit feedback** and why it needs different methods
8. **Hybrid systems**, and why production recommenders are never just one method
9. Diversity, popularity bias, and other practical failure modes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD, NMF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.neighbors import NearestNeighbors

rng = np.random.default_rng(seed=11)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

---
## 11.1 The utility matrix

Recommendation data is organised as a **utility matrix** $R$: rows are users, columns are
items, and each entry $R_{ui}$ is user $u$'s rating (or interaction) with item $i$.

The matrix is almost always **extremely sparse** — a user rates dozens of movies out of
thousands, buys hundreds of products out of millions. The task is to **predict the missing
entries**, then recommend the items with the highest predicted values that the user has not
already consumed.

Two data regimes:

- **Explicit feedback** — ratings (1–5 stars), thumbs up/down. Directly states preference, but
  users rarely give it, and what they say they like and what they actually watch can differ.
- **Implicit feedback** — clicks, purchases, watch time, dwell time. Abundant, but only ever
  positive: you observe what a user *did* do, never a reliable signal for what they actively
  dislike.

In [ ]:
# A small explicit-feedback utility matrix: movie ratings
np.set_printoptions(suppress=True)
movies = ["Inception", "Titanic", "Toy Story", "The Matrix", "Notebook",
         "Frozen", "Interstellar", "Up", "Die Hard", "Coco"]
users = [f"user{i}" for i in range(1, 9)]

# Simulate ratings from two latent tastes: sci-fi affinity and animated/family affinity
scifi_affinity = rng.normal(0, 1, len(users))
family_affinity = rng.normal(0, 1, len(users))
movie_scifi = np.array([0.9, -0.6, 0.1, 0.95, -0.7, 0.0, 0.9, 0.1, 0.3, 0.0])
movie_family = np.array([-0.2, 0.3, 0.95, -0.3, 0.4, 0.9, -0.1, 0.85, -0.4, 0.9])

true_pref = 3 + 1.4*np.outer(scifi_affinity, movie_scifi) + 1.4*np.outer(family_affinity, movie_family)
ratings_full = np.clip(np.round(true_pref + rng.normal(0, 0.35, true_pref.shape)), 1, 5)

# Make it sparse -- each user rates only some movies
mask = rng.random(ratings_full.shape) < 0.55
R = np.where(mask, ratings_full, np.nan)
R_df = pd.DataFrame(R, index=users, columns=movies)
print("Utility matrix (NaN = not rated):")
print(R_df.to_string())
print(f"\nDensity: {mask.mean():.1%} of entries are observed -- and this toy example is far")
print("denser than any real system (Netflix-scale sparsity is well under 1%).")

---
## 11.2 Content-based filtering

**Idea:** describe items by their **features** (genre, actors, ingredients, word content), and
recommend items similar to what the user has liked before. It uses only one user's own history
— no information about other users is needed.

**Advantages:** works immediately for new items (no cold start for items), explains itself
("because you liked X"), no dependence on other users' data.
**Disadvantages:** needs good item metadata, tends to over-specialise (recommends only more of
the same), and cannot help a user discover something outside their established taste.

In [ ]:
# Item features: genre tags per movie (a simple content representation)
genre_tags = {
    "Inception":    "scifi thriller mindbending",
    "Titanic":      "romance drama tragedy",
    "Toy Story":    "animated family adventure",
    "The Matrix":   "scifi action dystopian",
    "Notebook":     "romance drama",
    "Frozen":       "animated family musical",
    "Interstellar": "scifi drama space",
    "Up":           "animated family adventure",
    "Die Hard":     "action thriller",
    "Coco":         "animated family musical",
}
tag_docs = [genre_tags[m] for m in movies]
tfidf = TfidfVectorizer()
item_vectors = tfidf.fit_transform(tag_docs)
item_sim = cosine_similarity(item_vectors)
sim_df = pd.DataFrame(item_sim.round(3), index=movies, columns=movies)
print("Item-item content similarity (from genre tags):")
print(sim_df.to_string())

In [ ]:
def content_recommend(user, R_df, sim_df, n=3):
    '''Recommend items similar to the ones a user rated highly.'''
    rated = R_df.loc[user].dropna()
    liked = rated[rated >= 4].index
    if len(liked) == 0:
        return pd.Series(dtype=float)
    scores = sim_df[liked].mean(axis=1)
    scores = scores.drop(rated.index, errors="ignore")     # never re-recommend what they rated
    return scores.sort_values(ascending=False).head(n)

for u in ["user1", "user3"]:
    liked = R_df.loc[u].dropna()
    liked = liked[liked >= 4]
    print(f"\n{u} rated highly: {dict(liked)}")
    recs = content_recommend(u, R_df, sim_df)
    print(f"Content-based recommendations: {dict(recs.round(3))}")
print("\nBecause the similarity comes purely from GENRE TAGS, a user who liked sci-fi gets")
print("more sci-fi -- reliably relevant, but it will never surface something outside that")
print("lane. That is the over-specialisation weakness stated above, made concrete.")

---
## 11.3 Collaborative filtering

**Idea:** ignore item content entirely. Instead, use the **pattern of ratings** across many
users: if two users have rated things similarly in the past, they will probably agree on new
items too. "People like you also liked..."

### User-based

Find users similar to the target user (by their rating vectors), and recommend what those
similar users liked.

### Item-based

Find items similar to ones the user already rated highly — but here "similar" means *rated
similarly by the same users*, not similar in content. This is the version Amazon popularised
("customers who bought this also bought...") and it tends to be more stable, because item-item
similarities change more slowly than user-user ones as new users arrive.

In [ ]:
# User-based: cosine similarity on rating vectors, with missing values filled to the user's mean
R_filled = R_df.apply(lambda row: row.fillna(row.mean()), axis=1)
user_sim = pd.DataFrame(cosine_similarity(R_filled), index=users, columns=users)
print("User-user similarity (mean-filled ratings):")
print(user_sim.round(3).to_string())

def user_based_predict(user, item, R_df, user_sim, k=3):
    '''Weighted average of similar users' ratings for this item.'''
    others = R_df[item].dropna().drop(user, errors="ignore")
    if others.empty:
        return R_df.loc[user].mean()
    sims = user_sim.loc[user, others.index]
    top_k = sims.sort_values(ascending=False).head(k)
    if top_k.sum() <= 0:
        return others.mean()
    return (others[top_k.index] * top_k).sum() / top_k.sum()

target_user, target_item = "user1", "Frozen"
pred = user_based_predict(target_user, target_item, R_df, user_sim)
print(f"\nPredicted rating: {target_user} for '{target_item}' = {pred:.2f}")
print(f"(actual rating, if we had one: "
      f"{R_df.loc[target_user, target_item] if not pd.isna(R_df.loc[target_user, target_item]) else 'not rated'})")

In [ ]:
# Item-based: cosine similarity between ITEM rating vectors (across users)
item_rating_sim = pd.DataFrame(cosine_similarity(R_filled.T), index=movies, columns=movies)
print("Item-item similarity from RATING PATTERNS (not content!):")
print(item_rating_sim.round(3).to_string())

def item_based_predict(user, item, R_df, item_sim_ratings, k=3):
    rated = R_df.loc[user].dropna().drop(item, errors="ignore")
    if rated.empty:
        return R_df[item].mean()
    sims = item_sim_ratings.loc[item, rated.index]
    top_k = sims.sort_values(ascending=False).head(k)
    if top_k.sum() <= 0:
        return rated.mean()
    return (rated[top_k.index] * top_k).sum() / top_k.sum()

pred_item = item_based_predict(target_user, target_item, R_df, item_rating_sim)
print(f"\nItem-based prediction: {target_user} for '{target_item}' = {pred_item:.2f}")
print(f"User-based prediction was: {pred:.2f}")
print("\nCompare 'Interstellar' and 'The Matrix' by RATING pattern vs by CONTENT tags:")
print(f"  rating-pattern similarity : {item_rating_sim.loc['Interstellar', 'The Matrix']:.3f}")
print(f"  content-tag similarity    : {sim_df.loc['Interstellar', 'The Matrix']:.3f}")
print("Collaborative similarity can pick up relationships (a shared AUDIENCE, a shared")
print("release era, a shared director's fanbase) that plain genre tags never encode.")

---
## 11.4 Matrix factorisation

Neighbourhood methods (11.3) compare rows or columns of $R$ directly. **Matrix
factorisation** instead assumes ratings arise from a small number of hidden ("latent")
factors — taste dimensions like "prefers sci-fi" or "prefers family films", except the model
discovers them itself rather than being told:

$$R \approx P Q^\top, \qquad P \in \mathbb{R}^{n_{\text{users}}\times k},\
Q \in \mathbb{R}^{n_{\text{items}}\times k}$$

Each user gets a $k$-dimensional taste vector $p_u$; each item gets a $k$-dimensional
"appeal profile" $q_i$. The predicted rating is a dot product: $\hat{r}_{ui} = p_u^\top q_i$.

Two fitting strategies:

- **SVD-based** — works on the mean-filled (or otherwise imputed) matrix; fast, one-shot
- **ALS / SGD (gradient-based)** — optimise directly on the *observed* entries only, which is
  the mathematically correct approach for a sparse matrix (no need to fabricate the missing
  values first)

$k$ is a hyperparameter: too small underfits (cannot represent enough distinct tastes), too
large overfits (memorises noise in the sparse observations) — the identical trade-off from
every other model in this course.

In [ ]:
svd = TruncatedSVD(n_components=2, random_state=0)
P = svd.fit_transform(R_filled - R_filled.values.mean())
Q = svd.components_.T
pred_matrix = P @ Q.T + R_filled.values.mean()
pred_df = pd.DataFrame(pred_matrix, index=users, columns=movies)

print(f"Latent factors explain {svd.explained_variance_ratio_.sum():.1%} of the variance "
      f"with just k=2 components.\n")
print("User latent factors (taste vectors):")
print(pd.DataFrame(P, index=users, columns=["factor1", "factor2"]).round(3))
print("\nItem latent factors (appeal profiles):")
print(pd.DataFrame(Q, index=movies, columns=["factor1", "factor2"]).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(P[:, 0], P[:, 1], color="steelblue", s=80, label="users")
for u, (x, y) in zip(users, P):
    ax.annotate(u, (x, y), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.scatter(Q[:, 0], Q[:, 1], color="crimson", marker="^", s=100, label="movies")
for m, (x, y) in zip(movies, Q):
    ax.annotate(m, (x, y), fontsize=8, xytext=(4, 4), textcoords="offset points", color="crimson")
ax.axhline(0, color="grey", lw=0.6); ax.axvline(0, color="grey", lw=0.6)
ax.set_xlabel("latent factor 1"); ax.set_ylabel("latent factor 2")
ax.set_title("Users and items in the same latent taste space")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("Users near a movie in this space are predicted to like it. Nobody told the model")
print("that factor 1 correlates with 'sci-fi' or factor 2 with 'family' -- it discovered")
print("a low-dimensional structure purely from the pattern of ratings.")

In [ ]:
# Predictions and top-N recommendations for a user
for u in ["user1", "user5"]:
    already_rated = R_df.loc[u].dropna().index
    scores = pred_df.loc[u].drop(already_rated)
    print(f"\nTop 3 recommendations for {u} (predicted rating):")
    print(scores.sort_values(ascending=False).head(3).round(2).to_string())

In [ ]:
# ALS-style: optimise directly on OBSERVED entries only, via gradient descent
def matrix_factorization_sgd(R, k=2, n_epochs=200, lr=0.01, reg=0.05, seed=0):
    '''Train latent factors by SGD on observed entries only (the mathematically correct
    approach for a sparse matrix, unlike SVD on a mean-imputed dense copy).'''
    g = np.random.default_rng(seed)
    n_u, n_i = R.shape
    P = g.normal(0, 0.1, (n_u, k))
    Q = g.normal(0, 0.1, (n_i, k))
    obs = np.argwhere(~np.isnan(R))
    history = []
    for epoch in range(n_epochs):
        g.shuffle(obs)
        for u, i in obs:
            err = R[u, i] - P[u] @ Q[i]
            P[u] += lr * (err * Q[i] - reg * P[u])
            Q[i] += lr * (err * P[u] - reg * Q[i])
        preds = P @ Q.T
        rmse = np.sqrt(np.nanmean((R - preds) ** 2))
        history.append(rmse)
    return P, Q, history

P2, Q2, hist = matrix_factorization_sgd(R, k=2, n_epochs=150)
plt.plot(hist, color="steelblue")
plt.xlabel("epoch"); plt.ylabel("training RMSE (observed entries only)")
plt.title("Matrix factorisation by SGD, converging")
plt.show()

pred_sgd = pd.DataFrame(P2 @ Q2.T, index=users, columns=movies)
print(f"SVD-on-mean-fill training RMSE  : "
      f"{np.sqrt(np.nanmean((R - pred_df.values) ** 2)):.4f}")
print(f"SGD-on-observed-only training RMSE: {hist[-1]:.4f}")
print("\nSGD directly minimises error on what was actually observed, rather than first")
print("inventing values for the missing cells and then fitting to that invented matrix --")
print("which is why it is the standard approach at production scale (this is essentially")
print("what the Netflix Prize-winning methods did).")

---
## 11.5 The cold-start problem

Matrix factorisation and collaborative filtering both need **history** to work — a user with
zero ratings has no taste vector to learn, and a brand-new item has no appeal profile.

| Problem | Mitigation |
|---|---|
| **New user, no ratings** | Show popular/trending items; ask for a few preferences at signup; use demographic defaults |
| **New item, no ratings** | Use **content-based** features until enough interactions accumulate |
| **New user AND new item** | Pure popularity, or content similarity to items in a stated interest |

This is precisely why nearly every production recommender is a **hybrid** (section 11.8): pure
collaborative filtering is powerless exactly when it matters most — at onboarding.

In [ ]:
# Simulate the cold-start gap directly
brand_new_user = pd.Series(np.nan, index=movies, name="new_user")
print("A brand-new user has rated nothing:")
print(brand_new_user.to_string())

print(f"\nCollaborative filtering prediction: undefined -- there is no rating vector to")
print(f"compare against other users, and no row to feed into the factorisation.")

# Content-based needs at least one interaction; popularity needs none
popularity = R_df.count().sort_values(ascending=False)          # number of ratings received
avg_rating = R_df.mean().round(2)
print(f"\nFallback: most-rated items (a reasonable 'popular' proxy in this toy dataset):")
print(pd.DataFrame({"n_ratings": popularity, "avg_rating": avg_rating})
      .sort_values("n_ratings", ascending=False).head(5).to_string())
print("\nOnce this user rates even ONE movie highly, content-based recommendations become")
print("available immediately (section 11.2); collaborative filtering needs several ratings")
print("before its similarity estimates are trustworthy at all.")

---
## 11.6 Evaluating recommenders

Two very different questions, and they need different metrics.

### "How close are the predicted ratings?" — rating prediction

Standard regression metrics: **RMSE**, **MAE**, computed only on entries **held out** from
training (the same train/test discipline as every prior notebook — never evaluate on entries
the model saw).

### "How good is the top-N list?" — ranking quality

Nobody looks at a predicted rating of 3.7 directly; they look at the **top few items shown**.
Ranking metrics matter more in practice:

- **Precision@k** — of the top $k$ recommended, how many were actually relevant?
- **Recall@k** — of all relevant items, how many appeared in the top $k$?
- **Hit rate** — did at least one relevant item appear in the top $k$, for what fraction of
  users?

A model can have a mediocre RMSE and an excellent top-10 hit rate, or vice versa — they measure
different things, and the ranking metrics are usually the ones that matter to the business.

In [ ]:
# Honest RMSE: hide some observed entries, train on the rest, evaluate only on the hidden ones
observed = np.argwhere(~np.isnan(R))
train_idx, test_idx = train_test_split(observed, test_size=0.2, random_state=0)

R_train = R.copy()
for u, i in test_idx:
    R_train[u, i] = np.nan

P3, Q3, _ = matrix_factorization_sgd(R_train, k=2, n_epochs=200)
pred3 = P3 @ Q3.T

train_true = np.array([R[u, i] for u, i in train_idx])
train_pred = np.array([pred3[u, i] for u, i in train_idx])
test_true = np.array([R[u, i] for u, i in test_idx])
test_pred = np.array([pred3[u, i] for u, i in test_idx])

print(f"TRAINING RMSE (entries the model saw)   : "
      f"{np.sqrt(mean_squared_error(train_true, train_pred)):.4f}")
print(f"HELD-OUT RMSE (entries the model did NOT see): "
      f"{np.sqrt(mean_squared_error(test_true, test_pred)):.4f}")

global_mean = np.nanmean(R_train)
baseline_pred = np.full_like(test_true, global_mean)
print(f"Baseline (predict the global mean)      : "
      f"{np.sqrt(mean_squared_error(test_true, baseline_pred)):.4f}")
print("\nAs with every other model in this course: never trust the training-set error.")

In [ ]:
def precision_recall_at_k(R_true, R_pred, k=3, relevance_threshold=4):
    '''Mean precision@k and recall@k across users, using held-out ratings as ground truth.'''
    precisions, recalls = [], []
    for u in range(R_true.shape[0]):
        true_row = R_true[u]
        candidates = np.where(~np.isnan(true_row))[0]
        if len(candidates) == 0:
            continue
        relevant = set(candidates[true_row[candidates] >= relevance_threshold])
        if not relevant:
            continue
        ranked = candidates[np.argsort(-R_pred[u, candidates])]
        top_k = set(ranked[:k])
        precisions.append(len(top_k & relevant) / k)
        recalls.append(len(top_k & relevant) / len(relevant))
    return np.mean(precisions), np.mean(recalls)

# Build a "true" matrix containing ONLY the held-out test entries
R_test_only = np.full_like(R, np.nan)
for u, i in test_idx:
    R_test_only[u, i] = R[u, i]

for k in (1, 2, 3, 5):
    p_at_k, r_at_k = precision_recall_at_k(R_test_only, pred3, k=k)
    print(f"k={k}: precision@k = {p_at_k:.3f}   recall@k = {r_at_k:.3f}")
print("\nprecision@k answers 'of what we showed, how much was actually liked'.")
print("recall@k answers 'of everything they would have liked, how much did we surface'.")
print("Both matter, and (as with the precision/recall trade-off in Notebook 2) they")
print("trade off against each other as k grows.")

---
## 11.7 Implicit feedback

Explicit ratings are the exception in real systems; **implicit signals** (clicks, purchases,
watch-time, add-to-cart) are the norm. Implicit data has a fundamentally different structure:

- You only ever see **positive** signals (a click happened); you never see "the user actively
  disliked this" — absence of a click could mean dislike, or could mean they simply never saw
  the item.
- The scale is not a preference strength in the ratings sense; it is a **confidence** signal
  (watched 3 times = more confident they like it, not "rated it higher").
- Standard RMSE-style loss is the wrong tool, because there are no negative examples to
  regress against.

The standard approach — **Weighted/Alternating Least Squares for implicit feedback** (Hu, Koren
& Volinsky, 2008) — reframes the problem: treat every (user, item) pair as a binary preference
$p_{ui} \in \{0, 1\}$ (interacted or not), weighted by a confidence
$c_{ui} = 1 + \alpha \cdot (\text{interaction count})$. We do not have the full ALS machinery
in scikit-learn, so we implement the key idea directly: weighted matrix factorisation, which
generalises what we already built.

In [ ]:
# Simulate implicit feedback: view counts, only for items the user actually saw
view_counts = np.where(mask, rng.poisson(np.clip(true_pref - 2, 0.1, None)), 0)
print("Implicit signal (view counts), first 5 users:")
print(pd.DataFrame(view_counts, index=users, columns=movies).iloc[:5].to_string())

# Convert to preference (1 if any interaction) and confidence (grows with count)
pref = (view_counts > 0).astype(float)
alpha = 2.0
confidence = 1 + alpha * view_counts

def weighted_mf_sgd(pref, confidence, k=2, n_epochs=100, lr=0.02, reg=0.05, seed=0):
    '''Weighted matrix factorization for implicit feedback (all cells, weighted by confidence).'''
    g = np.random.default_rng(seed)
    n_u, n_i = pref.shape
    P = g.normal(0, 0.1, (n_u, k))
    Q = g.normal(0, 0.1, (n_i, k))
    idx = [(u, i) for u in range(n_u) for i in range(n_i)]
    for epoch in range(n_epochs):
        g.shuffle(idx)
        for u, i in idx:
            err = pref[u, i] - P[u] @ Q[i]
            w = confidence[u, i]
            P[u] += lr * (w * err * Q[i] - reg * P[u])
            Q[i] += lr * (w * err * P[u] - reg * Q[i])
    return P, Q

Pw, Qw = weighted_mf_sgd(pref, confidence, k=2)
implicit_scores = pd.DataFrame(Pw @ Qw.T, index=users, columns=movies)
print("\nImplicit-feedback preference scores (higher = more likely to engage):")
print(implicit_scores.round(2).iloc[:4].to_string())

In [ ]:
# Compare recommendations from explicit ratings vs implicit signals for the same user
u = "user2"
print(f"For {u}:")
print(f"  explicit-model top 3 : "
      f"{pred_df.loc[u].drop(R_df.loc[u].dropna().index).sort_values(ascending=False).head(3).round(2).to_dict()}")
print(f"  implicit-model top 3 : "
      f"{implicit_scores.loc[u].sort_values(ascending=False).head(3).round(2).to_dict()}")
print("\nThey can diverge: a user might RATE dramas highly (a stated preference) while")
print("actually SPENDING their time on action movies (a revealed preference). Which signal")
print("you trust depends on what you are optimising for -- and in most production systems,")
print("implicit feedback wins because it is abundant and reflects real behaviour rather")
print("than the occasional, possibly-aspirational, star rating.")

---
## 11.8 Hybrid systems

No production recommender relies on one method. The standard architecture blends signals:

| Approach | Handles cold-start? | Captures taste nuance? | Needs metadata? |
|---|---|---|---|
| Content-based | New items: yes. New users: no | Limited to described features | Yes |
| Collaborative | No | Rich, unstated patterns | No |
| Matrix factorisation | No | Rich, compact | No |
| Popularity | Yes | None | No |

**Common hybrid strategies:**

- **Weighted blend** — combine scores from several models (a business rule, or a small learned
  weighting)
- **Switching** — content-based for new users/items, collaborative once enough history exists
- **Cascade** — content-based to generate a candidate shortlist, collaborative filtering to
  rank it precisely
- **Feature-level** — feed content features and collaborative-filtering scores together into
  one supervised model (a gradient-boosted ranker, learning-to-rank)

In [ ]:
def hybrid_recommend(user, R_df, sim_content, pred_cf, n=3, cf_weight=0.6):
    '''Blend content-based and collaborative-filtering scores.'''
    rated = R_df.loc[user].dropna()
    liked = rated[rated >= 4].index
    content_scores = sim_content[liked].mean(axis=1) if len(liked) else pd.Series(0, index=movies)
    content_scores = (content_scores - content_scores.min()) / (content_scores.max() - content_scores.min() + 1e-9)

    cf_scores = pred_cf.loc[user]
    cf_scores = (cf_scores - cf_scores.min()) / (cf_scores.max() - cf_scores.min() + 1e-9)

    blended = cf_weight * cf_scores + (1 - cf_weight) * content_scores
    blended = blended.drop(rated.index, errors="ignore")
    return blended.sort_values(ascending=False).head(n)

for u in ["user1", "user7"]:
    print(f"\n{u}:")
    print(f"  content-only  : {dict(content_recommend(u, R_df, sim_df).round(3))}")
    print(f"  CF-only       : "
          f"{dict(pred_df.loc[u].drop(R_df.loc[u].dropna().index).sort_values(ascending=False).head(3).round(3))}")
    print(f"  hybrid (60/40): {dict(hybrid_recommend(u, R_df, sim_df, pred_df).round(3))}")
print("\nThe hybrid list draws on both signals, and the cf_weight parameter is a direct,")
print("tunable dial between 'trust the crowd' and 'trust this item's declared attributes'.")

---
## 11.9 Diversity, popularity bias, and other practical failure modes

Accuracy is not the only thing that matters in a shipped recommender.

**Popularity bias.** Collaborative filtering systematically favours popular items — they have
the most ratings, so their similarity and factor estimates are the most confident. This creates
a feedback loop: popular items get recommended more, get more interactions, get recommended
even more. Niche items and new creators struggle to ever surface.

**Filter bubbles / lack of diversity.** A model that only recommends near-neighbours of what you
already like can trap users in an increasingly narrow set of choices — good for short-term
click-through, often bad for long-term engagement and user satisfaction.

**The accuracy-diversity trade-off.** The single most "accurate" recommendation is often the
most obvious one. Deliberately re-ranking to inject variety typically costs a little measured
accuracy while improving real user outcomes.

**Serendipity.** The gold standard: recommendations that are relevant *and* surprising. Hardest
to measure, most valuable when achieved.

In [ ]:
# Popularity bias, visible directly in the toy dataset
n_ratings = R_df.count().sort_values(ascending=False)
print("How often each movie gets recommended in the top-3 hybrid list, across all users:")
from collections import Counter
rec_counts = Counter()
for u in users:
    for item in hybrid_recommend(u, R_df, sim_df, pred_df, n=3).index:
        rec_counts[item] += 1

pop_vs_rec = pd.DataFrame({"n_ratings_received": n_ratings,
                           "times_recommended": pd.Series(rec_counts)}).fillna(0)
print(pop_vs_rec.sort_values("n_ratings_received", ascending=False).to_string())
print(f"\nCorrelation between how OFTEN an item was rated and how OFTEN it gets")
print(f"recommended: {pop_vs_rec.corr().iloc[0,1]:.3f}")
print("\nEven in this small toy example, items with more historical ratings tend to get")
print("recommended more -- their latent factors and similarities are simply better")
print("estimated. At web scale this effect concentrates attention on a small head of")
print("popular items and starves the long tail.")

In [ ]:
def diversify(recs_series, sim_matrix, lambda_=0.5, n=3):
    '''Maximal Marginal Relevance: trade relevance against dissimilarity to already-picked items.'''
    candidates = list(recs_series.index)
    picked = []
    while candidates and len(picked) < n:
        def mmr_score(item):
            relevance = recs_series[item]
            if not picked:
                return relevance
            max_sim = max(sim_matrix.loc[item, p] for p in picked)
            return lambda_ * relevance - (1 - lambda_) * max_sim
        best = max(candidates, key=mmr_score)
        picked.append(best)
        candidates.remove(best)
    return picked

u = "user1"
raw_scores = pred_df.loc[u].drop(R_df.loc[u].dropna().index)
plain_top3 = raw_scores.sort_values(ascending=False).head(3).index.tolist()
diverse_top3 = diversify(raw_scores, sim_df, lambda_=0.5, n=3)

print(f"Plain top-3 (pure predicted rating)      : {plain_top3}")
print(f"Diversified top-3 (MMR, lambda=0.5)      : {diverse_top3}")
print(f"\nContent similarity WITHIN the plain list      : "
      f"{sim_df.loc[plain_top3, plain_top3].values[np.triu_indices(3, 1)].mean():.3f}")
print(f"Content similarity WITHIN the diversified list : "
      f"{sim_df.loc[diverse_top3, diverse_top3].values[np.triu_indices(3, 1)].mean():.3f}")
print("\nMMR (Maximal Marginal Relevance) explicitly penalises recommending items too")
print("similar to ones already selected, trading a small amount of pure predicted-rating")
print("accuracy for a list that covers more of the user's plausible interests.")

---
## Exercises

**Exercise 1.** Using the MovieLens-style toy matrix `R`, implement and compare user-based and
item-based collaborative filtering for **every** missing entry, then measure which method has
lower RMSE against a held-out split (reuse the split from section 11.6).

In [ ]:
# --- Solution 1 -------------------------------------------------------------
def cf_predict_all(R_arr, method="item", k=3):
    R_local = pd.DataFrame(R_arr, index=users, columns=movies)
    filled = R_local.apply(lambda row: row.fillna(row.mean()), axis=1)
    if method == "user":
        sim = pd.DataFrame(cosine_similarity(filled), index=users, columns=users)
        preds = np.zeros_like(R_arr)
        for ui, u in enumerate(users):
            for ii, item in enumerate(movies):
                preds[ui, ii] = user_based_predict(u, item, R_local, sim, k=k)
    else:
        sim = pd.DataFrame(cosine_similarity(filled.T), index=movies, columns=movies)
        preds = np.zeros_like(R_arr)
        for ui, u in enumerate(users):
            for ii, item in enumerate(movies):
                preds[ui, ii] = item_based_predict(u, item, R_local, sim, k=k)
    return preds

user_preds = cf_predict_all(R_train, method="user")
item_preds = cf_predict_all(R_train, method="item")

for name, preds in [("user-based CF", user_preds), ("item-based CF", item_preds)]:
    test_p = np.array([preds[u, i] for u, i in test_idx])
    rmse = np.sqrt(mean_squared_error(test_true, test_p))
    mae = mean_absolute_error(test_true, test_p)
    print(f"  {name:<16} held-out RMSE = {rmse:.4f}   MAE = {mae:.4f}")
print(f"  {'matrix factorization':<16} held-out RMSE = "
      f"{np.sqrt(mean_squared_error(test_true, test_pred)):.4f}")
print(f"  {'global mean baseline':<16} held-out RMSE = "
      f"{np.sqrt(mean_squared_error(test_true, baseline_pred)):.4f}")
print("\nOn a dataset this small and dense, the methods are close; item-based CF is")
print("usually the more STABLE of the two neighbourhood methods in production, because")
print("item-item similarities change more slowly than user-user ones as the user base grows.")

**Exercise 2.** Vary the number of latent factors $k$ in matrix factorisation from 1 to 6.
Plot training RMSE and held-out RMSE against $k$, identify the point of overfitting, and relate
it to the bias-variance trade-off from the statistics module.

In [ ]:
# --- Solution 2 -------------------------------------------------------------
ks = range(1, 7)
train_rmses, test_rmses = [], []
for k in ks:
    Pk, Qk, _ = matrix_factorization_sgd(R_train, k=k, n_epochs=200, reg=0.05)
    predk = Pk @ Qk.T
    tr_p = np.array([predk[u, i] for u, i in train_idx])
    te_p = np.array([predk[u, i] for u, i in test_idx])
    train_rmses.append(np.sqrt(mean_squared_error(train_true, tr_p)))
    test_rmses.append(np.sqrt(mean_squared_error(test_true, te_p)))

plt.plot(list(ks), train_rmses, "o-", color="steelblue", label="training RMSE")
plt.plot(list(ks), test_rmses, "o-", color="crimson", label="held-out RMSE")
best_k = list(ks)[int(np.argmin(test_rmses))]
plt.axvline(best_k, color="black", ls="--", label=f"best k = {best_k}")
plt.xlabel("number of latent factors (k)"); plt.ylabel("RMSE")
plt.title("More latent factors: training error keeps falling, held-out error does not")
plt.legend(fontsize=8); plt.show()

print(f"{'k':>4}{'train RMSE':>12}{'held-out RMSE':>15}")
for k, tr, te in zip(ks, train_rmses, test_rmses):
    print(f"{k:>4}{tr:>12.4f}{te:>15.4f}")
print(f"\nBest k by held-out RMSE: {best_k}")
print("\nThis is the identical bias-variance shape as the polynomial-degree and tree-depth")
print("curves from earlier in the course: too few factors underfits (the model cannot")
print("represent enough distinct tastes -- HIGH BIAS), too many factors memorises the sparse")
print("observed entries (HIGH VARIANCE), with 'k' playing exactly the role that 'degree'")
print("or 'max_depth' played there. The regularisation term (reg=0.05) also matters here,")
print("in the same way alpha did for ridge regression.")

**Exercise 3.** Build a cold-start scenario: add a brand-new user who has rated exactly one
movie, "Interstellar", with a 5. Compare what content-based, collaborative-filtering, and
hybrid recommenders produce for them, and comment on which is trustworthy given only one data
point.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
new_user_ratings = pd.Series(np.nan, index=movies, name="new_user9")
new_user_ratings["Interstellar"] = 5.0

R_extended = pd.concat([R_df, new_user_ratings.to_frame().T])
print("Extended matrix (new user added):")
print(R_extended.tail(2).to_string())

# Content-based: works immediately from the one rating
content_new = content_recommend("new_user9", R_extended, sim_df, n=4)
print(f"\nContent-based recommendations: {dict(content_new.round(3))}")

# Collaborative filtering: refit with the new row included
R_ext_filled = R_extended.apply(lambda row: row.fillna(row.mean()), axis=1)
R_ext_arr = R_extended.to_numpy()
P_new, Q_new, _ = matrix_factorization_sgd(R_ext_arr, k=2, n_epochs=200)
pred_new = pd.DataFrame(P_new @ Q_new.T, index=R_extended.index, columns=movies)
cf_new = pred_new.loc["new_user9"].drop("Interstellar").sort_values(ascending=False).head(4)
print(f"\nCollaborative-filtering recommendations (refit with 1 new rating):")
print(dict(cf_new.round(3)))

hybrid_new = hybrid_recommend("new_user9", R_extended, sim_df, pred_new, n=4)
print(f"\nHybrid recommendations: {dict(hybrid_new.round(3))}")

print("\nWhich is trustworthy with only one rating:")
print("  * CONTENT-BASED is the most defensible here: it reasons directly and transparently")
print("    from 'you liked a sci-fi/space movie' to 'here is another one' -- exactly the")
print("    right amount of confidence for one data point.")
print("  * COLLABORATIVE FILTERING had almost nothing to learn from -- the new user's row")
print("    was mostly the row mean before fitting, so its taste vector is barely")
print("    distinguishable from an 'average user', and its recommendations should be")
print("    treated with real suspicion until more ratings arrive.")
print("  * The HYBRID list leans on content for exactly this reason (cf_weight can be")
print("    lowered for low-history users specifically, a common production pattern:")
print("    ramp UP the collaborative weight as a user's history grows).")

**Exercise 4 (challenge).** Design an evaluation for a hybrid recommender being considered for
production: propose both an offline evaluation plan (with specific metrics and a train/test
protocol that avoids leakage) and an online evaluation plan (an A/B test design), and explain
what each catches that the other cannot.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
print("=" * 74)
print("OFFLINE EVALUATION PLAN")
print("=" * 74)
print("1. SPLIT PROTOCOL (avoid leakage):")
print("   - Use a TIME-BASED split, not a random one: train on interactions before")
print("     date T, evaluate on interactions after T. A random split lets the model")
print("     see a user's FUTURE behaviour when predicting their present -- the same")
print("     leakage warned about for time series (Notebook 10) applies here, because")
print("     recommendation is inherently sequential (today's taste is informed by")
print("     yesterday's watches).")
print("   - Within the test period, hold out a user's LAST k interactions specifically")
print("     ('leave-last-k-out') to simulate the real deployment question: given")
print("     history up to now, what comes next?")
print("   - Cold-start slice: hold out a set of users/items with FEWER than 3")
print("     historical interactions and evaluate them separately -- the aggregate")
print("     metric hides how badly (or well) the system handles exactly the case")
print("     that matters most for growth.")
print()
print("2. METRICS (report all, not just one):")
print("   - RMSE / MAE on held-out explicit ratings (if available) -- calibration check")
print("   - precision@10, recall@10, hit-rate@10 -- what the user will actually SEE")
print("   - catalog coverage: what fraction of the catalog ever appears in a top-10 list")
print("     across all users -- the direct measure of popularity bias from section 11.9")
print("   - intra-list diversity: mean pairwise dissimilarity within each user's list")
print("   - performance broken out by user history length (cold, warm, hot) separately")

In [ ]:
print("=" * 74)
print("ONLINE EVALUATION PLAN (A/B TEST)")
print("=" * 74)
print("1. DESIGN:")
print("   - Randomise at the USER level (not the impression level), so a user sees a")
print("     consistent experience -- the same 'don't shuffle' discipline from the")
print("     sampling module (statistics Notebook 4), applied to assignment rather")
print("     than to a train/test split.")
print("   - Fix the sample size IN ADVANCE with a power calculation on the primary")
print("     metric (statistics Notebook 6) -- no peeking, no early stopping on a")
print("     promising trend.")
print("   - Pre-register the primary metric (one number the launch decision hinges")
print("     on) and a short list of guardrail metrics that must not regress.")
print()
print("2. METRICS:")
print("   PRIMARY (pick one, tied to business value):")
print("     - click-through rate on recommended items, or")
print("     - downstream conversion / watch-completion / purchase rate")
print("   GUARDRAILS (must not get worse):")
print("     - session length / return rate (catches short-term-CTR-optimised, long-term-")
print("       harmful filter bubbles that no offline metric would flag)")
print("     - catalog coverage in PRODUCTION traffic (offline coverage can look fine while")
print("       production serving logic re-introduces popularity bias, e.g. via caching)")
print("     - latency (a fancier model that is too slow is not shippable regardless of")
print("       accuracy -- the same lesson as the SVM latency case study, Notebook 7)")
print()
print("3. ANALYSIS:")
print("   - Two-sample test (or CI) on the primary metric between control and treatment,")
print("     reported with an effect size, not just a p-value (statistics Notebook 6 and 7)")
print("   - Segment the effect by user cohort (new vs returning, cold vs warm) -- an")
print("     average lift can hide a system that helps power users and actively hurts")
print("     new ones")
print("=" * 74)
print("WHY BOTH ARE NEEDED")
print("=" * 74)
print("Offline evaluation is fast, cheap, and reproducible, but it can only ever answer")
print("'how well would this have predicted what already happened?' -- it cannot see")
print("how users respond to items they were never shown, so it systematically favours")
print("models that resemble the status quo (the same LOGGED-DATA bias that plagues")
print("offline evaluation of any interactive system).")
print()
print("Online evaluation is the only way to measure genuine causal effect on user")
print("behaviour, but it is slow, costly in engineering effort, and risks real user")
print("harm if the model is badly broken -- which is exactly why it should run AFTER")
print("a model has already cleared the offline bar, on a fraction of traffic, with a")
print("clear rollback plan.")
print()
print("The two are not redundant: offline evaluation catches broken models cheaply")
print("before they ever reach a user; online evaluation catches models that look good")
print("on paper but change user behaviour in ways no offline metric anticipated")
print("(filter bubbles, novelty fatigue, unexpected guardrail regressions). A")
print("production launch decision should never rest on offline metrics alone.")

---
## Summary

| Concept | Key point |
|---|---|
| Utility matrix | Sparse users × items grid; the object every method tries to fill in |
| Content-based | Recommend by item feature similarity; no user cold-start help |
| Collaborative filtering | Recommend by user/item behavioural similarity; no cold-start help at all |
| User-based vs item-based | Item-based is usually more stable in production |
| Matrix factorisation | $R \approx PQ^\top$; latent taste factors, learned end-to-end |
| $k$ (number of factors) | The bias-variance knob; tune it exactly like polynomial degree |
| Cold start | The core weakness of every behavioural method; mitigate with content or popularity |
| RMSE / MAE | Rating-prediction accuracy; compute only on held-out entries |
| Precision@k / recall@k / hit rate | Ranking quality — usually what the business cares about |
| Implicit feedback | Positive-only, confidence-weighted; needs weighted factorisation, not plain regression |
| Hybrid systems | Blend, switch, cascade, or feature-combine — always used in production |
| Popularity bias | Collaborative methods favour well-known items by construction |
| MMR / diversification | Trade a little relevance for coverage of the user's real interests |
| Offline vs online eval | Offline is cheap but biased toward the status quo; online measures real causal effect |

---

## 🎓 Module complete

**Machine Learning Fundamentals and Predictive Analytics** — eleven notebooks, from a straight
line to a full recommender system:

| # | Notebook | What it added |
|---|---|---|
| 1 | [Linear Regression](1.%20Linear%20Regression.ipynb) | Predicting a number, and reading what a model learned |
| 2 | [Logistic Regression](2.%20Logistic%20Regression.ipynb) | Predicting a probability and a category |
| 3 | [Decision Trees](3.%20Decision%20Trees.ipynb) | Non-linear boundaries you can read as a flowchart |
| 4 | [K-Nearest Neighbors](4.%20K-Nearest%20Neighbors.ipynb) | A model with no training phase at all |
| 5 | [Clustering](5.%20Clustering.ipynb) | Learning structure with no labels |
| 6 | [Random Forest](6.%20Random%20Forest.ipynb) | Averaging many trees into a strong default |
| 7 | [Support Vector Machines](7.%20Support%20Vector%20Machines.ipynb) | Maximum-margin boundaries and the kernel trick |
| 8 | [Naive Bayes](8.%20Naive%20Bayes.ipynb) | A fast, tiny-data, high-dimensional classifier |
| 9 | [Introduction to NLP](9.%20Introduction%20to%20NLP.ipynb) | Turning text into features models can use |
| 10 | [Time Series Analytics](10.%20Time%20Series%20Analytics.ipynb) | Forecasting when order is the signal |
| 11 | [Recommender Systems](11.%20Recommender%20Systems.ipynb) | Predicting preference, not just outcome |

Every notebook here rests on the [Statistical Foundations for Data Science](../6.%20Statistical%20Foundations%20for%20Data%20Science)
module — probability, sampling, hypothesis testing, and above all honest evaluation. That
discipline, not any single algorithm, is the real throughline of both modules.